# RAG retrieval evaluation — audited and debugged

**Purpose.** Reproduce the stored retrieval evaluation without modifying the original notebook, then determine why semantic, hybrid, and language-aware hybrid produce near-identical aggregate metrics.

**Safety contract.** Existing project files are immutable. Offline cells only read versioned source/CSV files. They do not load `.env`, connect to PostgreSQL, generate embeddings, download models, or call paid APIs. Live database/model cells are disabled by default and visibly labelled `NOT RUN`; they contain only read-only `SELECT` instrumentation.

Status labels used throughout: `CONFIRMED BY CODE`, `CONFIRMED BY EXECUTION`, `INFERENCE`, `INCONCLUSIVE`, `NOT RUN`.

Created from a read-only audit on 2026-08-06. The original notebook and prior results are not edited.

## 1. Purpose, scope, and safety constraints

This notebook has four goals: (1) preserve the original metric definitions as a baseline; (2) compare stored top-5 outputs at query level; (3) recompute independent evidence-aware metrics; and (4) provide guarded instrumentation for raw semantic, lexical, language-filter and RRF behavior when a safe configured environment is available.

No conclusion about raw lexical scores or database candidate pools is made from stored CSVs because those values were not persisted for hybrid strategies.

## 2. Environment and reproducibility metadata

In [ ]:
from __future__ import annotations

import ast
import csv
import importlib.metadata
import json
import math
import os
import platform
import random
import re
import statistics
import sys
from collections import Counter, defaultdict
from pathlib import Path

SEED = 20260806
random.seed(SEED)
try:
    import numpy as np
    np.random.seed(SEED)
except Exception:
    np = None

ROOT = Path.cwd().resolve()
if not (ROOT / 'eval').exists() and (ROOT.parent / 'eval').exists():
    ROOT = ROOT.parent
if not (ROOT / 'eval' / 'gold_references.csv').exists():
    raise FileNotFoundError('Run from the repository root or notebooks/ directory.')

def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return 'NOT_INSTALLED'

metadata = {
    'analysis_seed': SEED,
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'repository_root': str(ROOT),
    'packages': {name: package_version(name) for name in [
        'pandas', 'numpy', 'scipy', 'langdetect', 'psycopg2', 'pgvector',
        'sentence-transformers', 'python-dotenv'
    ]},
    # Non-secret values only. Secret variables are represented only by presence flags.
    'non_secret_environment': {name: os.getenv(name) for name in [
        'EMBED_MODEL_NAME', 'LLM_MODEL_NAME', 'TOP_K'
    ] if os.getenv(name) is not None},
    'secret_presence_only': {name: bool(os.getenv(name)) for name in [
        'TELEGRAM_TOKEN', 'HF_API_KEY', 'OPENAI_API_KEY', 'PGPASSWORD'
    ]},
}
print(json.dumps(metadata, indent=2, ensure_ascii=False))

## 3. Map of the original pipeline

### Inputs and evaluation unit

- `eval/gold_references.csv`: 77 evidence rows grouped into 31 unique `question_id` values. Each evidence supplies `expected_doc`, `expected_page`, and `expected_text`.
- The evaluated retrieval unit is a **retrieved chunk**, reduced for relevance to `(basename(source), page_num)`. Chunk IDs and text are not used by the original relevance calculation.
- Document relevance ignores page; page relevance requires the same basename and page within `±page_tolerance` (default 1).

### Strategy call paths — CONFIRMED BY CODE

| Strategy | Notebook evaluator | Project function | Candidate generation and rank | Filters | Returned score |
|---|---|---|---|---|---|
| semantic | `evaluate_search` | `app.retrieval.search` | pgvector cosine distance `<=>`; direct `ORDER BY`; `LIMIT k` | optional topic, not supplied | `distance` |
| hybrid | `evaluate_hybrid_search` | `app.retrieval.hybrid_search` | semantic top `4k` + full-text top `4k`; RRF `1/(60+rank)`; final `LIMIT k` | optional topic, not supplied; FTS config `simple` | RRF `score` |
| language-aware hybrid | `evaluate_language_aware_hybrid_search` | `app.retrieval.language_aware_hybrid_search` | same two candidate branches and RRF | detected `lang` applied to both branches; PostgreSQL config maps es/en/fr/de | RRF `score` |

There is no score normalization, weighted linear fusion, reranking, or deduplication step. SQL joins candidates by `(doc_id, chunk_id)`. Ties have no deterministic secondary key.

### Original metric definitions

- `Hit@k = int(any(relevance_by_rank))`.
- `Binary Recall@k = int(sum(relevance_by_rank) > 0)`: mathematically identical to Hit@k.
- `Precision@k = number of relevant retrieved chunks / configured k`, even if fewer than `k` are returned.
- MRR uses the first relevant retrieved chunk.
- Aggregation is an unweighted macro mean over 31 questions.

In [ ]:
# Static call-path validation without importing project modules.
def notebook_code(path):
    nb = json.loads(path.read_text(encoding='utf-8'))
    return '\n'.join(''.join(c.get('source', [])) for c in nb['cells'] if c.get('cell_type') == 'code')

def calls_inside_function(source, function_name):
    tree = ast.parse(source)
    fn = next(n for n in tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and n.name == function_name)
    names = []
    for node in ast.walk(fn):
        if isinstance(node, ast.Call):
            target = node.func
            if isinstance(target, ast.Name):
                names.append(target.id)
            elif isinstance(target, ast.Attribute):
                names.append(target.attr)
    return sorted(set(names))

nb_source = notebook_code(ROOT / 'notebooks' / 'RAG-evaluation.ipynb')
retrieval_source = (ROOT / 'app' / 'retrieval.py').read_text(encoding='utf-8')
call_path_check = {
    'evaluate_search': calls_inside_function(nb_source, 'evaluate_search'),
    'evaluate_hybrid_search': calls_inside_function(nb_source, 'evaluate_hybrid_search'),
    'evaluate_language_aware_hybrid_search': calls_inside_function(nb_source, 'evaluate_language_aware_hybrid_search'),
    'search': calls_inside_function(retrieval_source, 'search'),
    'hybrid_search': calls_inside_function(retrieval_source, 'hybrid_search'),
    'language_aware_hybrid_search': calls_inside_function(retrieval_source, 'language_aware_hybrid_search'),
}
print(json.dumps(call_path_check, indent=2))
assert 'search' in call_path_check['evaluate_search']
assert 'hybrid_search' in call_path_check['evaluate_hybrid_search']
assert 'language_aware_hybrid_search' in call_path_check['evaluate_language_aware_hybrid_search']

## 4. Baseline reproduction

**Label: BASELINE REPRODUCTION.** The following cells copy the original relevance and aggregation semantics, but read the already stored outputs rather than calling the database. This reproduces the published numbers without changing data or requiring services.

In [ ]:
STRATEGIES = ('semantic', 'hybrid', 'language_aware_hybrid')
RESULT_FILES = {
    'semantic': ROOT / 'eval' / 'retrieval_semantic_search_evaluation_results.csv',
    'hybrid': ROOT / 'eval' / 'retrieval_hybrid_search_evaluation_results.csv',
    'language_aware_hybrid': ROOT / 'eval' / 'retrieval_language_aware_hybrid_search_evaluation_results.csv',
}

def read_csv_rows(path):
    with path.open(encoding='utf-8-sig', newline='') as stream:
        return list(csv.DictReader(stream))

def literal_list(value):
    parsed = ast.literal_eval(value)
    return list(parsed)

def normalize_doc_name(value):
    return Path(str(value)).name.strip() if value is not None else ''

def normalize_page(value):
    if value is None:
        return ''
    value = str(value).strip()
    try:
        return str(int(float(value)))
    except (TypeError, ValueError):
        return value

def unique_in_order(values):
    return list(dict.fromkeys(values))

RESULT_ROWS = {name: read_csv_rows(path) for name, path in RESULT_FILES.items()}
RESULT_BY_QID = {name: {row['question_id']: row for row in rows} for name, rows in RESULT_ROWS.items()}
GOLD_ROWS = read_csv_rows(ROOT / 'eval' / 'gold_references.csv')
GOLD_BY_QID = defaultdict(list)
for row in GOLD_ROWS:
    GOLD_BY_QID[row['question_id']].append(row)
QUESTION_IDS = sorted(GOLD_BY_QID)
assert all(set(RESULT_BY_QID[name]) == set(QUESTION_IDS) for name in STRATEGIES)
print({name: len(rows) for name, rows in RESULT_ROWS.items()}, 'gold_evidences=', len(GOLD_ROWS))

In [ ]:
def original_is_relevant_doc(retrieved_pair, gold_pairs):
    return any(retrieved_pair[0] == gold_doc for gold_doc, _ in gold_pairs)

def original_is_relevant_page(retrieved_pair, gold_pairs, page_tolerance=1):
    doc, page = retrieved_pair
    try:
        page = int(page)
    except (TypeError, ValueError):
        return False
    for gold_doc, gold_page in gold_pairs:
        try:
            gold_page = int(gold_page)
        except (TypeError, ValueError):
            continue
        if doc == gold_doc and abs(page - gold_page) <= page_tolerance:
            return True
    return False

def original_row_metrics(row, k=5, page_tolerance=1):
    gold_pairs = literal_list(row['gold_doc_pages'])
    retrieved = literal_list(row['retrieved_doc_pages_ranked'])[:k]
    doc_hits = [original_is_relevant_doc(pair, gold_pairs) for pair in retrieved]
    page_hits = [original_is_relevant_page(pair, gold_pairs, page_tolerance) for pair in retrieved]
    first_doc = next((rank for rank, hit in enumerate(doc_hits, 1) if hit), None)
    first_page = next((rank for rank, hit in enumerate(page_hits, 1) if hit), None)
    return {
        f'doc_hit@{k}': int(any(doc_hits)),
        f'page_hit@{k}': int(any(page_hits)),
        f'doc_precision@{k}': sum(doc_hits) / k,
        f'page_precision@{k}': sum(page_hits) / k,
        f'doc_binary_recall@{k}': int(sum(doc_hits) > 0),
        f'page_binary_recall@{k}': int(sum(page_hits) > 0),
        'doc_reciprocal_rank': 0 if first_doc is None else 1 / first_doc,
        'page_reciprocal_rank': 0 if first_page is None else 1 / first_page,
        'returned_count': len(retrieved),
    }

def macro_means(records, keys):
    return {key: statistics.mean(record[key] for record in records) for key in keys}

BASELINE_KEYS = [
    'doc_hit@5', 'doc_precision@5', 'doc_binary_recall@5', 'doc_reciprocal_rank',
    'page_hit@5', 'page_precision@5', 'page_binary_recall@5', 'page_reciprocal_rank',
]
BASELINE = {}
for strategy in STRATEGIES:
    records = [original_row_metrics(RESULT_BY_QID[strategy][qid], 5, 1) for qid in QUESTION_IDS]
    BASELINE[strategy] = macro_means(records, BASELINE_KEYS)

# Independent comparison with the saved aggregate file.
stored_aggregate = read_csv_rows(ROOT / 'eval' / 'retrieval_combined_evaluation_metrics.csv')
metric_name_map = {
    'Doc Hit@5': 'doc_hit@5', 'Doc Precision@5': 'doc_precision@5',
    'Doc Binary Recall@5': 'doc_binary_recall@5', 'Doc MRR': 'doc_reciprocal_rank',
    'Page Hit@5': 'page_hit@5', 'Page Precision@5': 'page_precision@5',
    'Page Binary Recall@5': 'page_binary_recall@5', 'Page MRR': 'page_reciprocal_rank',
}
stored_column = {
    'semantic': 'semantic_search',
    'hybrid': 'hybrid_search',
    'language_aware_hybrid': 'language_aware_hybrid_search',
}
for row in stored_aggregate:
    key = metric_name_map[row['metric']]
    for strategy in STRATEGIES:
        assert math.isclose(BASELINE[strategy][key], float(row[stored_column[strategy]]), rel_tol=0, abs_tol=1e-12)
print('CONFIRMED BY EXECUTION: all 24 aggregate strategy/metric values reproduced exactly.')
for metric in BASELINE_KEYS:
    print(metric, {s: round(BASELINE[s][metric], 6) for s in STRATEGIES})

### Baseline result — CONFIRMED BY EXECUTION

The offline recomputation exactly reproduces all values in `retrieval_combined_evaluation_metrics.csv`. Semantic and hybrid have identical aggregate metrics; language-aware changes precision/MRR but not either binary Hit/Recall aggregate.

## 5. Diagnostic hypotheses

Each hypothesis below has code evidence, a diagnostic, an expected observation, an actual result, and a conclusion. Conclusions based on stored CSV execution are limited to the persisted document-page top-5 outputs.

In [ ]:
HYPOTHESES = [
 {'id':'H01','hypothesis':'All branches call the same function/effective code path','evidence':'Original cells 15/21/26 call search, hybrid_search, language_aware_hybrid_search respectively.','test':'Static AST call-path extraction.','expected_if_true':'Same called function or alias.','actual':'Three distinct functions and SQL constructions.','conclusion':'REJECTED — CONFIRMED BY CODE'},
 {'id':'H02','hypothesis':'Strategy arguments are ignored, overwritten, or misassigned','evidence':'Each evaluator passes question and TOP_K directly; topic is intentionally omitted.','test':'Inspect calls and parameter use.','expected_if_true':'A common function/constant overrides strategy or k.','actual':'k is used; no common strategy flag exists.','conclusion':'REJECTED — CONFIRMED BY CODE'},
 {'id':'H03','hypothesis':'Cached previous retrieval results are reused across strategies','evidence':'Each evaluator constructs a fresh list/DataFrame and invokes retrieval per question.','test':'Trace object creation and compare result-file hashes/content.','expected_if_true':'Shared cache/output object or identical file reference.','actual':'No retrieval-result cache; three distinct stored files.','conclusion':'REJECTED — CONFIRMED BY CODE'},
 {'id':'H04','hypothesis':'Mutable DataFrames/result objects are shared','evidence':'Separate local retrieval_rows lists and separate DataFrame names.','test':'Static scope/object trace.','expected_if_true':'One list/DataFrame reused or mutated.','actual':'Objects are separately constructed.','conclusion':'REJECTED — CONFIRMED BY CODE'},
 {'id':'H05','hypothesis':'Semantic and lexical branches produce the same effective top-k set','evidence':'Only final hybrid output is stored; raw branches are absent.','test':'Compare final top-5 doc-page sequences; live candidate probe required for raw branches.','expected_if_true':'Most final sets/rankings identical.','actual':'Semantic vs hybrid rankings and sets identical for 30/31 queries.','conclusion':'CONFIRMED at persisted doc-page top-5 granularity; raw candidates INCONCLUSIVE'},
 {'id':'H06','hypothesis':'Lexical scores are zero/missing/constant/too weak','evidence':'Hybrid CSV omits lexical scores and raw candidate rows.','test':'Guarded read-only candidate SQL and lexical-score distribution.','expected_if_true':'Zero/constant scores or no lexical candidates.','actual':'No persisted values; live test not run.','conclusion':'NOT TESTED'},
 {'id':'H07','hypothesis':'RRF preserves the semantic ranking','evidence':'RRF k=60 and candidate pools 4k in retrieval.py.','test':'Compare semantic vs hybrid top-5 and inspect raw ranks live.','expected_if_true':'Nearly all rankings identical.','actual':'30/31 doc-page rankings identical; only Q006 diverges at rank 1.','conclusion':'CONFIRMED BY EXECUTION at persisted granularity'},
 {'id':'H08','hypothesis':'Language detection returns the same language/fallback for most queries','evidence':'Detected language is not stored.','test':'Run seeded langdetect and record per query.','expected_if_true':'Mostly es or a common fallback.','actual':'langdetect unavailable in audit runtime; no stored field.','conclusion':'INCONCLUSIVE'},
 {'id':'H09','hypothesis':'Language filter is not applied or ineffective','evidence':'SQL adds lang=%(query_lang)s to semantic and lexical branches when detection succeeds.','test':'Compare language-aware outputs and empty result counts.','expected_if_true':'Outputs always identical to hybrid.','actual':'12/31 rankings differ; Q006, Q018, Q030 return zero results.','conclusion':'REJECTED as no-effect; CONFIRMED brittle behavior'},
 {'id':'H10','hypothesis':'Language metadata is absent or inconsistent','evidence':'Ingestion derives lang from filenames; no DB metadata snapshot is stored.','test':'Read-only SELECT distribution and null/unknown checks.','expected_if_true':'Nulls, unexpected codes, or no rows for detected codes.','actual':'Corpus filenames contain only es/en; DB not inspected.','conclusion':'INCONCLUSIVE'},
 {'id':'H11','hypothesis':'Deduplication removes differences','evidence':'No deduplication in retrieval SQL or evaluation; repeated docs/pages remain.','test':'Search code and count duplicates.','expected_if_true':'Explicit distinct/dedupe stage.','actual':'No stage; duplicates are common.','conclusion':'REJECTED — CONFIRMED BY CODE'},
 {'id':'H12','hypothesis':'All strategies are evaluated from one stored retrieval output','evidence':'Three evaluator calls and three distinct result files.','test':'Compare hashes, sequences and per-query differences.','expected_if_true':'Same source path/object.','actual':'Separate outputs; language-aware differs on 12 queries.','conclusion':'REJECTED — CONFIRMED BY CODE/EXECUTION'},
 {'id':'H13','hypothesis':'Identifier normalization creates false matches','evidence':'Document path is reduced to basename; page cast to integer; tolerance is ±1.','test':'Check basename collisions, page formats, and strict/tolerant metrics.','expected_if_true':'Collisions or invalid page strings.','actual':'No current filename collision; all gold pages parse as integers. Tolerance still broadens relevance.','conclusion':'REJECTED for collisions; tolerance effect requires explicit reporting'},
 {'id':'H14','hypothesis':'Exact/binary relevance hides partial evidence coverage','evidence':'Hit/precision operate on doc/page matches and ignore expected_text/evidence IDs.','test':'Compute true recall over all unique gold docs/pages/evidences.','expected_if_true':'Coverage metrics materially below binary Hit.','actual':'At k=5 semantic Doc Hit=.774 but mean true doc recall=.611 and evidence coverage=.430.','conclusion':'CONFIRMED BY EXECUTION'},
 {'id':'H15','hypothesis':'Hit@5 and Binary Recall@5 are redundant','evidence':'Both are int(any hit).','test':'Symbolic comparison and all-row equality.','expected_if_true':'No mismatches.','actual':'Zero mismatches in all three strategies.','conclusion':'CONFIRMED BY CODE/EXECUTION'},
 {'id':'H16','hypothesis':'Gold has one relevant document/page per question','evidence':'77 evidence rows for 31 questions.','test':'Count unique gold docs/pages per qid.','expected_if_true':'All counts equal one.','actual':'Only 16/31 have one unique doc and 15/31 one unique doc-page.','conclusion':'REJECTED — CONFIRMED BY EXECUTION'},
 {'id':'H17','hypothesis':'k=5 is too coarse to reveal ranking differences','evidence':'Only common cross-strategy stored cutoff is k=5.','test':'Evaluate k=1,3,5,10 per strategy.','expected_if_true':'Conclusions vary materially with k.','actual':'Stored semantic Hit changes across k; hybrid/language-aware top-10 rows absent.','conclusion':'INCONCLUSIVE cross-strategy; sensitivity CONFIRMED for semantic'},
 {'id':'H18','hypothesis':'31 questions provide insufficient power','evidence':'Binary metric resolution is 1/31=.0323; bootstrap intervals are wide.','test':'Deterministic bootstrap CIs and paired tests.','expected_if_true':'CIs overlap and paired effects are small/non-significant.','actual':'Observed language-aware deltas have small effects and paired p>=.25.','conclusion':'CONFIRMED methodological limitation'},
 {'id':'H19','hypothesis':'Aggregate means conceal query-level differences','evidence':'Language-aware changes 12 rankings.','test':'Per-query delta table and first divergence.','expected_if_true':'Differences cancel or do not cross binary thresholds.','actual':'Q023 gains Doc Hit while Q030 loses it; Page Hit changes on zero queries.','conclusion':'CONFIRMED BY EXECUTION'},
 {'id':'H20','hypothesis':'Queries do not discriminate lexical/multilingual behavior','evidence':'All questions appear Spanish; corpus/gold mostly Spanish; 22/31 have a simple exact lexical anchor proxy.','test':'Seeded language detection plus anchor/cross-language challenge-set analysis.','expected_if_true':'Homogeneous language and high exact overlap.','actual':'Corpus 11 es/7 en; 68/77 gold evidences es; only 5 queries reference any English gold.','conclusion':'INFERENCE; challenge set required'},
]
for row in HYPOTHESES:
    print(row['id'], row['conclusion'], '-', row['actual'])

## 6. Strategy-dispatch and call-path validation

**CONFIRMED BY CODE:** the branches are distinct. Similarity is not caused by calling the same Python function. However, semantic and hybrid share the same embedding model/table, and hybrid RRF uses a large `rrf_k=60`, so a lexical branch must substantially alter candidate membership/rank to affect top-5.

In [ ]:
# Availability of persisted diagnostic fields.
stored_field_availability = {}
for strategy, rows in RESULT_ROWS.items():
    stored_field_availability[strategy] = sorted(rows[0].keys())
print(json.dumps(stored_field_availability, indent=2))

semantic_context = []
for row in RESULT_ROWS['semantic']:
    for item in literal_list(row['retrieved_context']):
        semantic_context.append({
            'question_id': row['question_id'], 'query': row['question'],
            'rank': item.get('rank'), 'doc_id': item.get('doc_id'),
            'chunk_id': item.get('chunk_id'), 'source': item.get('source'),
            'lang': item.get('lang'), 'topic': item.get('topic'),
            'raw_semantic_distance': item.get('distance'),
            'raw_lexical_score': None, 'normalized_semantic_score': None,
            'normalized_lexical_score': None, 'fusion_score': item.get('score'),
        })
print('semantic rows with chunk+distance:', len(semantic_context))
print('hybrid/language-aware raw candidate scores: NOT STORED')

## 7. Candidate, score, and ranking analysis

The next offline cell compares persisted final rankings. The guarded live cell beneath it can recover raw semantic/lexical candidates and RRF scores, but remains `NOT RUN` unless explicitly enabled in a configured environment. Enabling model loading may download a model; do not enable without authorization.

In [ ]:
def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if a | b else 1.0

def overlap_at_k(a, b, k):
    return len(set(a[:k]) & set(b[:k])) / k

def first_divergence(a, b):
    for index in range(max(len(a), len(b))):
        if index >= len(a) or index >= len(b) or a[index] != b[index]:
            return index + 1
    return None

def rank_biased_overlap(a, b, p=0.9):
    depth = max(len(a), len(b))
    left, right, total, agreement = set(), set(), 0.0, 0.0
    for d in range(1, depth + 1):
        if d <= len(a): left.add(a[d-1])
        if d <= len(b): right.add(b[d-1])
        agreement = len(left & right) / d
        total += agreement * p ** (d - 1)
    return (1-p) * total + agreement * p ** depth

def spearman_on_common(a, b):
    ra = {value: rank for rank, value in enumerate(unique_in_order(a), 1)}
    rb = {value: rank for rank, value in enumerate(unique_in_order(b), 1)}
    common = list(ra.keys() & rb.keys())
    n = len(common)
    if n < 2:
        return None
    return 1 - 6 * sum((ra[x]-rb[x])**2 for x in common) / (n * (n*n-1))

def persisted_pairs(strategy, qid, k=5):
    return literal_list(RESULT_BY_QID[strategy][qid]['retrieved_doc_pages_ranked'])[:k]

PAIRWISE_SUMMARY = []
for left, right in [('semantic','hybrid'), ('semantic','language_aware_hybrid'), ('hybrid','language_aware_hybrid')]:
    rows = []
    for qid in QUESTION_IDS:
        a, b = persisted_pairs(left, qid), persisted_pairs(right, qid)
        rows.append({
            'question_id': qid, 'identical_ranking': a == b,
            'identical_set': set(a) == set(b), 'jaccard': jaccard(a,b),
            'overlap@5': overlap_at_k(a,b,5), 'rbo_p0.9': rank_biased_overlap(a,b),
            'spearman_common': spearman_on_common(a,b),
            'first_divergence_rank': first_divergence(a,b),
        })
    summary = {
        'pair': f'{left} vs {right}',
        'identical_rankings': sum(r['identical_ranking'] for r in rows),
        'identical_sets': sum(r['identical_set'] for r in rows),
        'mean_jaccard': statistics.mean(r['jaccard'] for r in rows),
        'mean_overlap@5': statistics.mean(r['overlap@5'] for r in rows),
        'mean_rbo_p0.9': statistics.mean(r['rbo_p0.9'] for r in rows),
        'different_qids': [r['question_id'] for r in rows if not r['identical_ranking']],
    }
    PAIRWISE_SUMMARY.append(summary)
for row in PAIRWISE_SUMMARY:
    print(json.dumps(row, ensure_ascii=False, indent=2))

In [ ]:
# OPTIONAL LIVE READ-ONLY CANDIDATE PROBE — NOT RUN BY DEFAULT.
# Do not set both flags without explicit authorization: project embedding code may download a model.
ENABLE_LIVE_DB = False
ALLOW_MODEL_LOADING = False
LIVE_TOP_K_VALUES = [1, 3, 5, 10]

def live_candidate_probe(query, strategy='hybrid', top_k=5, rrf_k=60):
    if not ENABLE_LIVE_DB or not ALLOW_MODEL_LOADING:
        return {'status':'NOT RUN', 'reason':'ENABLE_LIVE_DB and ALLOW_MODEL_LOADING are False'}
    if strategy not in {'semantic','hybrid','language_aware_hybrid'}:
        raise ValueError(strategy)
    # Imports occur only after the explicit guards. app.config may load .env indirectly; values are never printed.
    from app.config import settings
    from app.db import get_conn
    from app.embeddings import embed_query
    from app.retrieval import SUPPORTED_LANGS, _safe_detect_lang

    query_lang = _safe_detect_lang(query) if strategy == 'language_aware_hybrid' else None
    ts_config = SUPPORTED_LANGS[query_lang] if query_lang is not None else 'simple'
    if ts_config not in {'simple','english','spanish','french','german'}:
        raise ValueError('Unexpected text-search config')
    vector = embed_query(settings.embed_model_name, query)
    pool = top_k * 4
    language_clause = ' AND lang = %(lang)s' if query_lang is not None else ''
    semantic_sql = f'''
        SELECT doc_id, chunk_id, source, page_num, lang, topic,
               embedding <=> %(emb)s::vector AS semantic_distance
        FROM rag_chunks
        WHERE TRUE {language_clause}
        ORDER BY embedding <=> %(emb)s::vector
        LIMIT %(pool)s
    '''
    lexical_sql = f'''
        SELECT doc_id, chunk_id, source, page_num, lang, topic,
               ts_rank_cd(to_tsvector('{ts_config}', content), q) AS lexical_score
        FROM rag_chunks, plainto_tsquery('{ts_config}', %(query)s) q
        WHERE to_tsvector('{ts_config}', content) @@ q {language_clause}
        ORDER BY lexical_score DESC
        LIMIT %(pool)s
    '''
    for statement in (semantic_sql, lexical_sql):
        assert statement.lstrip().upper().startswith(('SELECT','WITH','EXPLAIN'))
    params = {'emb': vector, 'query': query, 'lang': query_lang, 'pool': pool}
    conn = get_conn()
    try:
        conn.set_session(readonly=True, autocommit=False)
        with conn.cursor() as cursor:
            cursor.execute(semantic_sql, params)
            semantic = cursor.fetchall()
            lexical = []
            if strategy != 'semantic':
                cursor.execute(lexical_sql, params)
                lexical = cursor.fetchall()
        conn.rollback()
    finally:
        conn.close()

    fields = ['doc_id','chunk_id','source','page_num','lang','topic','raw_score']
    sem = [dict(zip(fields,row), semantic_rank=i) for i,row in enumerate(semantic,1)]
    lex = [dict(zip(fields,row), lexical_rank=i) for i,row in enumerate(lexical,1)]
    merged = {}
    for row in sem:
        key=(row['doc_id'],row['chunk_id']); merged.setdefault(key,{}).update(row)
        merged[key]['raw_semantic_distance']=merged[key].pop('raw_score')
    for row in lex:
        key=(row['doc_id'],row['chunk_id']); merged.setdefault(key,{}).update(row)
        merged[key]['raw_lexical_score']=merged[key].pop('raw_score')
    for row in merged.values():
        row.setdefault('raw_semantic_distance',None); row.setdefault('raw_lexical_score',None)
        row.setdefault('semantic_rank',None); row.setdefault('lexical_rank',None)
        row['normalized_semantic_score']=None  # Original code performs no score normalization.
        row['normalized_lexical_score']=None
        row['rrf_score']=(0 if row['semantic_rank'] is None else 1/(rrf_k+row['semantic_rank'])) + (0 if row['lexical_rank'] is None else 1/(rrf_k+row['lexical_rank']))
    ranked = sorted(merged.values(), key=lambda row: row['rrf_score'], reverse=True)[:top_k] if strategy != 'semantic' else sem[:top_k]
    return {'status':'EXECUTED READ ONLY','strategy':strategy,'detected_language':query_lang,'ts_config':ts_config,'semantic_candidates':sem,'lexical_candidates':lex,'final':ranked}

print(live_candidate_probe('diagnostic guard check'))

## 8. Query-level comparison

This table is the primary explanation for aggregate similarity. It identifies every query whose top-5 sequence or metric differs. `candidate_diff_but_same_topk` cannot be known offline because pre-fusion candidates were not stored.

In [ ]:
QUERY_DIAGNOSTICS = []
metric_columns = ['doc_hit@5','doc_precision@5','doc_reciprocal_rank','page_hit@5','page_precision@5','page_reciprocal_rank']
for qid in QUESTION_IDS:
    sequences = {s: persisted_pairs(s,qid,5) for s in STRATEGIES}
    stored_metrics = {s: {m: float(RESULT_BY_QID[s][qid][m]) for m in metric_columns} for s in STRATEGIES}
    row = {
        'question_id': qid, 'query': GOLD_BY_QID[qid][0]['question'],
        'semantic_vs_hybrid_same_rank': sequences['semantic'] == sequences['hybrid'],
        'semantic_vs_language_same_rank': sequences['semantic'] == sequences['language_aware_hybrid'],
        'semantic_hybrid_first_divergence': first_divergence(sequences['semantic'],sequences['hybrid']),
        'semantic_language_first_divergence': first_divergence(sequences['semantic'],sequences['language_aware_hybrid']),
        'semantic_hybrid_jaccard': jaccard(sequences['semantic'],sequences['hybrid']),
        'semantic_language_jaccard': jaccard(sequences['semantic'],sequences['language_aware_hybrid']),
        'language_returned': len(sequences['language_aware_hybrid']),
        'metric_differences': [m for m in metric_columns if len({stored_metrics[s][m] for s in STRATEGIES}) > 1],
        'candidate_diff_but_same_topk': 'UNKNOWN — raw candidate pools not stored',
    }
    QUERY_DIAGNOSTICS.append(row)

print('Queries with semantic/hybrid ranking differences:', [r['question_id'] for r in QUERY_DIAGNOSTICS if not r['semantic_vs_hybrid_same_rank']])
print('Queries with language-aware ranking differences:', [r['question_id'] for r in QUERY_DIAGNOSTICS if not r['semantic_vs_language_same_rank']])
print('Queries with any aggregate-component difference:', [(r['question_id'],r['metric_differences']) for r in QUERY_DIAGNOSTICS if r['metric_differences']])
print('Queries with zero language-aware results:', [r['question_id'] for r in QUERY_DIAGNOSTICS if r['language_returned']==0])

## 9. Independent metric validation

**Label: PROPOSED CORRECTION.** The baseline remains unchanged above. The alternative definitions below deduplicate documents and expose numerators/denominators. Page/evidence coverage counts how many unique gold `(document,page)` items are covered within the original ±1 tolerance.

In [ ]:
def dcg(labels):
    return sum(label / math.log2(rank + 1) for rank, label in enumerate(labels, 1))

def average_precision(labels, total_relevant):
    if total_relevant == 0:
        return None
    hits = 0
    total = 0.0
    for rank, label in enumerate(labels, 1):
        if label:
            hits += 1
            total += hits / rank
    return total / total_relevant

def independent_metrics(strategy, qid, k, page_tolerance=1):
    retrieved = [(normalize_doc_name(d), normalize_page(p)) for d,p in persisted_pairs(strategy,qid,k)]
    gold_pairs = unique_in_order([(normalize_doc_name(r['expected_doc']), normalize_page(r['expected_page'])) for r in GOLD_BY_QID[qid]])
    gold_docs = unique_in_order([doc for doc,_ in gold_pairs])
    returned_docs = unique_in_order([doc for doc,_ in retrieved])
    doc_labels = [int(doc in gold_docs) for doc,_ in retrieved]
    page_labels = []
    covered_gold_pairs = set()
    for doc, page in retrieved:
        hit = False
        try: page_i = int(page)
        except ValueError: page_i = None
        for index,(gold_doc,gold_page) in enumerate(gold_pairs):
            try: gold_page_i = int(gold_page)
            except ValueError: gold_page_i = None
            if doc == gold_doc and page_i is not None and gold_page_i is not None and abs(page_i-gold_page_i) <= page_tolerance:
                hit = True
                covered_gold_pairs.add(index)
        page_labels.append(int(hit))
    unique_relevant_docs = len(set(returned_docs) & set(gold_docs))
    ideal_doc = sorted(doc_labels, reverse=True)[:min(k,len(gold_docs))]
    ideal_page = sorted(page_labels, reverse=True)[:min(k,len(gold_pairs))]
    first_doc = next((i for i,v in enumerate(doc_labels,1) if v),None)
    first_page = next((i for i,v in enumerate(page_labels,1) if v),None)
    return {
      'question_id':qid,'strategy':strategy,'k':k,
      'returned_numerator_doc_chunk_hits':sum(doc_labels),'returned_denominator':len(retrieved),
      'fixed_k_denominator':k,'unique_relevant_docs_numerator':unique_relevant_docs,'gold_docs_denominator':len(gold_docs),
      'covered_gold_pairs_numerator':len(covered_gold_pairs),'gold_pairs_denominator':len(gold_pairs),
      'doc_hit':int(any(doc_labels)),'page_hit':int(any(page_labels)),
      'original_doc_precision_fixed_k':sum(doc_labels)/k,'original_page_precision_fixed_k':sum(page_labels)/k,
      'doc_precision_returned':sum(doc_labels)/len(retrieved) if retrieved else 0,
      'unique_doc_precision':unique_relevant_docs/len(returned_docs) if returned_docs else 0,
      'true_doc_recall':unique_relevant_docs/len(gold_docs) if gold_docs else None,
      'evidence_coverage':len(covered_gold_pairs)/len(gold_pairs) if gold_pairs else None,
      'doc_mrr':0 if first_doc is None else 1/first_doc,'page_mrr':0 if first_page is None else 1/first_page,
      'doc_ndcg':dcg(doc_labels)/(dcg(ideal_doc) or 1),'page_ndcg':dcg(page_labels)/(dcg(ideal_page) or 1),
      'doc_map':average_precision(doc_labels,len(gold_docs)),'page_map':average_precision(page_labels,len(gold_pairs)),
      'no_relevant_doc':int(not any(doc_labels)),'no_relevant_page':int(not any(page_labels)),
    }

INDEPENDENT_PER_QUERY = [independent_metrics(s,qid,k) for k in [1,3,5] for s in STRATEGIES for qid in QUESTION_IDS]
for k in [1,3,5]:
    print('k=',k)
    for strategy in STRATEGIES:
        rows=[r for r in INDEPENDENT_PER_QUERY if r['k']==k and r['strategy']==strategy]
        keys=['doc_hit','original_doc_precision_fixed_k','unique_doc_precision','true_doc_recall','doc_mrr','doc_ndcg','doc_map','page_hit','original_page_precision_fixed_k','evidence_coverage','page_mrr','page_ndcg','page_map','no_relevant_doc','no_relevant_page']
        print(strategy,{key:round(statistics.mean(r[key] for r in rows if r[key] is not None),6) for key in keys})

In [ ]:
# Duplicate-impact audit and available k=10 evidence.
for strategy in STRATEGIES:
    duplicate_doc_rows = 0
    duplicate_pair_rows = 0
    for qid in QUESTION_IDS:
        pairs = persisted_pairs(strategy,qid,5)
        docs = [doc for doc,_ in pairs]
        duplicate_doc_rows += len(set(docs)) < len(docs)
        duplicate_pair_rows += len(set(pairs)) < len(pairs)
    print(strategy, 'duplicate_doc_rows=',duplicate_doc_rows, 'duplicate_doc_page_rows=',duplicate_pair_rows)

semantic_k_table = read_csv_rows(ROOT / 'eval' / 'retrieval_semantic_search_evaluation_metrics_by_top_k.csv')
print('Stored semantic-only aggregate k sensitivity:')
for row in semantic_k_table:
    print(row)
print('Hybrid and language-aware per-query k=10: NOT RUN / NOT STORED. Use the guarded live probe only with authorization.')

## 10. Sensitivity and statistical analysis

Bootstrap and permutation procedures use the fixed seed above. Intervals describe uncertainty over this 31-question sample; they do not repair selection bias or incomplete gold labels.

In [ ]:
gold_structure = []
for qid in QUESTION_IDS:
    rows=GOLD_BY_QID[qid]
    pairs=unique_in_order([(normalize_doc_name(r['expected_doc']),normalize_page(r['expected_page'])) for r in rows])
    docs=unique_in_order([d for d,_ in pairs])
    langs=sorted({Path(r['expected_doc']).stem.split('_')[1] for r in rows})
    gold_structure.append({'question_id':qid,'evidences':len(rows),'unique_docs':len(docs),'unique_doc_pages':len(pairs),'gold_doc_languages':langs})
print('questions=',len(QUESTION_IDS),'evidences=',len(GOLD_ROWS),'binary_metric_resolution=',1/len(QUESTION_IDS))
for key in ['evidences','unique_docs','unique_doc_pages']:
    print(key,dict(sorted(Counter(row[key] for row in gold_structure).items())))

corpus_language=Counter()
for path in (ROOT/'docs').rglob('*.pdf'):
    parts=path.stem.split('_'); corpus_language[parts[1] if len(parts)>1 else 'unknown']+=1
print('document_language_distribution_from_filenames=',dict(corpus_language))
print('queries_with_any_english_gold=',[r['question_id'] for r in gold_structure if 'en' in r['gold_doc_languages']])
print('Actual detected query language: NOT STORED / NOT RUN')

# Explicit lexical-anchor proxy; this is a heuristic, not the PostgreSQL lexical score.
STOPWORDS=set('que cual cuales como cuando donde por para con sin del las los una uno unos unas este esta estos estas sobre entre desde hasta puede pueden debo debe se es son y o en de la el un al mi mis su sus lo mas ya si'.split())
def tokens(text):
    return [t for t in re.findall(r'[a-záéíóúüñ0-9]+',text.casefold()) if len(t)>=3 and t not in STOPWORDS]
anchor_qids=[]
for qid in QUESTION_IDS:
    qtokens=tokens(GOLD_BY_QID[qid][0]['question'])
    gold_tokens=set(tokens(' '.join(row['expected_text'] for row in GOLD_BY_QID[qid])))
    if any(len(token)>=5 and token in gold_tokens for token in qtokens): anchor_qids.append(qid)
print('lexical_anchor_proxy=',len(anchor_qids),'of',len(QUESTION_IDS),'queries')

def bootstrap_ci(values, seed=SEED, samples=10000):
    rng=random.Random(seed);n=len(values)
    means=sorted(statistics.mean(values[rng.randrange(n)] for _ in range(n)) for _ in range(samples))
    return means[int(.025*samples)],means[int(.975*samples)]

def paired_sign_flip_pvalue(differences, seed=SEED):
    nonzero=[d for d in differences if abs(d)>1e-15]; observed=abs(statistics.mean(differences))
    if not nonzero:return 1.0
    if len(nonzero)<=20:
        simulated=[abs(sum(d if mask>>i&1 else -d for i,d in enumerate(nonzero))/len(differences)) for mask in range(1<<len(nonzero))]
    else:
        rng=random.Random(seed)
        simulated=[abs(sum(d if rng.random()<.5 else -d for d in nonzero)/len(differences)) for _ in range(100000)]
    return sum(value>=observed-1e-15 for value in simulated)/len(simulated)

STATISTICAL_RESULTS=[]
metric_lookup={'doc_hit':'doc_hit','doc_precision':'original_doc_precision_fixed_k','doc_mrr':'doc_mrr','page_hit':'page_hit','page_precision':'original_page_precision_fixed_k','page_mrr':'page_mrr'}
for label,key in metric_lookup.items():
    values={s:[independent_metrics(s,qid,5)[key] for qid in QUESTION_IDS] for s in STRATEGIES}
    for s in STRATEGIES:
        low,high=bootstrap_ci(values[s])
        STATISTICAL_RESULTS.append({'metric':label,'comparison':s,'mean':statistics.mean(values[s]),'delta':None,'ci95_low':low,'ci95_high':high,'paired_p':None,'effect_dz':None})
    for other in ['hybrid','language_aware_hybrid']:
        diffs=[b-a for a,b in zip(values['semantic'],values[other])]
        sd=statistics.stdev(diffs) if len(set(diffs))>1 else 0
        STATISTICAL_RESULTS.append({'metric':label,'comparison':f'semantic vs {other}','mean':None,'delta':statistics.mean(diffs),'ci95_low':None,'ci95_high':None,'paired_p':paired_sign_flip_pvalue(diffs),'effect_dz':statistics.mean(diffs)/sd if sd else 0})
for row in STATISTICAL_RESULTS:
    if row['comparison'].startswith('semantic vs'):
        print(row)

In [ ]:
# Paired bootstrap confidence intervals for strategy deltas at k=5.
def paired_bootstrap_ci(left, right, seed=SEED, samples=10000):
    if len(left) != len(right):
        raise ValueError('Paired samples must have equal length.')
    rng = random.Random(seed)
    n = len(left)
    deltas = sorted(
        statistics.mean(right[i] - left[i] for i in (rng.randrange(n) for _ in range(n)))
        for _ in range(samples)
    )
    return deltas[int(.025 * samples)], deltas[int(.975 * samples)]

PAIRED_INTERVALS = []
for label, key in metric_lookup.items():
    values = {s: [independent_metrics(s, qid, 5)[key] for qid in QUESTION_IDS] for s in STRATEGIES}
    for other in ['hybrid', 'language_aware_hybrid']:
        low, high = paired_bootstrap_ci(values['semantic'], values[other])
        diffs = [b - a for a, b in zip(values['semantic'], values[other])]
        row = {
            'metric': label,
            'comparison': f'{other} minus semantic',
            'mean_delta': statistics.mean(diffs),
            'paired_bootstrap_ci95': (low, high),
            'paired_p': paired_sign_flip_pvalue(diffs),
        }
        PAIRED_INTERVALS.append(row)
        print(row)

## 11. Proposed corrections and controlled experiments

1. **Metric correction:** keep baseline unchanged, but add deduplicated document precision, true document recall, evidence coverage, nDCG and MAP.
2. **Language fallback experiment:** because language-aware returned zero rows for Q006, Q018 and Q030, simulate a conservative fallback to ordinary hybrid only when the language-filtered result is empty. This is an offline counterfactual, not a rerun of retrieval.
3. **Raw-score experiment:** use the guarded live probe to record query language, candidate membership, distances, lexical scores, ranks and RRF score. `NOT RUN`.
4. **Cutoff experiment:** run all three strategies at k=1,3,5,10 through the read-only live path. `NOT RUN` because hybrid/language-aware top-10 outputs were not persisted.
5. **Challenge-set experiment:** add—but do not mix into the frozen baseline—predeclared lexical, multilingual, typo, abbreviation and cross-language queries. No speculative score is reported here.

In [ ]:
# Controlled offline counterfactual: fallback to hybrid only when language-aware returned no rows.
FALLBACK_ROWS = {}
for qid in QUESTION_IDS:
    language_row = dict(RESULT_BY_QID['language_aware_hybrid'][qid])
    if not literal_list(language_row['retrieved_doc_pages_ranked']):
        language_row = dict(RESULT_BY_QID['hybrid'][qid])
        language_row['question_id'] = qid
        language_row['fallback_applied'] = 'hybrid_after_empty_language_result'
    else:
        language_row['fallback_applied'] = ''
    FALLBACK_ROWS[qid]=language_row

fallback_original=[original_row_metrics(FALLBACK_ROWS[qid],5,1) for qid in QUESTION_IDS]
fallback_baseline=macro_means(fallback_original,BASELINE_KEYS)
print('fallback qids=',[qid for qid,row in FALLBACK_ROWS.items() if row['fallback_applied']])
for key in BASELINE_KEYS:
    print(key,'before=',round(BASELINE['language_aware_hybrid'][key],6),'after_simulated_fallback=',round(fallback_baseline[key],6),'delta=',round(fallback_baseline[key]-BASELINE['language_aware_hybrid'][key],6))

## 12. Before/after comparison

The baseline is never overwritten. “After” values are either corrected metric definitions over the same stored rankings or the explicitly labelled empty-result fallback simulation. They are not claims about a rerun retrieval system.

In [ ]:
BEFORE_AFTER=[]
for strategy in STRATEGIES:
    rows=[independent_metrics(strategy,qid,5) for qid in QUESTION_IDS]
    BEFORE_AFTER.append({
      'strategy':strategy,
      'baseline_doc_hit@5':statistics.mean(r['doc_hit'] for r in rows),
      'baseline_chunk_precision@5':statistics.mean(r['original_doc_precision_fixed_k'] for r in rows),
      'corrected_unique_doc_precision@5':statistics.mean(r['unique_doc_precision'] for r in rows),
      'corrected_true_doc_recall@5':statistics.mean(r['true_doc_recall'] for r in rows),
      'corrected_evidence_coverage@5':statistics.mean(r['evidence_coverage'] for r in rows),
      'doc_ndcg@5':statistics.mean(r['doc_ndcg'] for r in rows),
      'doc_map@5':statistics.mean(r['doc_map'] for r in rows),
    })
for row in BEFORE_AFTER:print({k:(round(v,6) if isinstance(v,float) else v) for k,v in row.items()})
print('Simulated language-aware empty-result fallback:',{k:round(v,6) for k,v in fallback_baseline.items()})

## 13. Conclusions and prioritized recommendations

1. **CONFIRMED BY EXECUTION:** semantic and hybrid are genuinely almost identical at the persisted document-page top-5 level: 30/31 rankings and sets are identical. Q006 is the only divergence; neither ranking retrieves the gold document/page, so aggregate metrics do not change.
2. **CONFIRMED BY CODE:** semantic and hybrid are not the same implementation. The similarity therefore reflects ineffective differentiation on this query set/fusion setup, not accidental common dispatch.
3. **CONFIRMED BY EXECUTION:** language-aware changes 12/31 rankings, but aggregate binary metrics conceal this. Q023 gains Doc Hit while Q030 loses it, exactly cancelling; Page Hit changes for no query.
4. **INFERENCE:** Q006, Q018 and Q030 returning zero language-aware rows is consistent with supported-language misclassification into a language absent from the es/en corpus. Detected language was not logged, so the precise cause is not confirmed.
5. **CONFIRMED BY CODE/EXECUTION:** Hit and Binary Recall are the same metric. Original “Doc Precision” counts relevant chunks, including repeated documents, rather than unique documents.
6. **CONFIRMED BY EXECUTION:** binary Hit overstates coverage in a multi-evidence gold set: at k=5 semantic Doc Hit is .774, mean true document recall is .611, and page-evidence coverage is .430.
7. **CONFIRMED methodological limitation:** n=31 gives .0323 resolution for binary means, wide bootstrap intervals, and small paired effects; the current set is mostly Spanish and lexically anchored.
8. **INCONCLUSIVE:** whether lexical scores are absent/constant/weak, and whether RRF k=60 or candidate pool size causes rank preservation, requires the guarded raw-candidate probe.

### Five highest-priority next actions

1. Run the read-only raw candidate probe after explicitly authorizing model loading and confirming a configured DB; record detected language and all branch scores.
2. Add an empty-language-result fallback and validate language codes/metadata before claiming language-aware gains.
3. Replace Binary Recall with true recall over unique gold evidence and report deduplicated precision, nDCG and MAP alongside Hit/MRR.
4. Evaluate all three strategies at k=1,3,5,10 and report paired per-query deltas with intervals/effect sizes.
5. Create a frozen discriminatory challenge set for lexical anchors, abbreviations, typos, multilingual and cross-language retrieval; keep it separate from the existing baseline.

## 14. Execution manifest

In [ ]:
import hashlib

AUDIT_TIMESTAMP = '2026-08-06T13:31:27+02:00'
TOOL_VERSIONS = {
    'PowerShell': '5.1.26100.8875',
    'Git': '2.51.0.windows.2',
    'ripgrep': '15.2.0',
    'Python': sys.version.split()[0],
}
READ_ONLY_COMMANDS_USED = [
    'git status --short --branch',
    'Get-ChildItem (path/name enumeration only; .env not opened)',
    'rg --files and rg -n static source searches (excluding .env and __pycache__)',
    'Python -B JSON/AST validation',
    'Python -B sequential in-memory execution of offline-safe notebook cells',
    'Get-FileHash -Algorithm SHA256 / hashlib.sha256 read-only hashing',
]
ANALYZED_FILES = [
    ROOT / 'notebooks' / 'RAG-evaluation.ipynb',
    ROOT / 'app' / 'retrieval.py',
    ROOT / 'app' / 'db.py',
    ROOT / 'app' / 'config.py',
    ROOT / 'scripts' / 'create_tables.sql',
    ROOT / 'scripts' / 'ingest_docs.py',
    ROOT / 'requirements.txt',
    ROOT / 'eval' / 'gold_references.csv',
    ROOT / 'eval' / 'retrieval_semantic_search_evaluation_results.csv',
    ROOT / 'eval' / 'retrieval_hybrid_search_evaluation_results.csv',
    ROOT / 'eval' / 'retrieval_language_aware_hybrid_search_evaluation_results.csv',
    ROOT / 'eval' / 'retrieval_combined_evaluation_metrics.csv',
    ROOT / 'eval' / 'retrieval_semantic_search_evaluation_metrics_by_top_k.csv',
]
missing_files = [str(path.relative_to(ROOT)) for path in ANALYZED_FILES if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f'Analyzed-file manifest paths missing: {missing_files}')
FILE_SHA256 = {
    str(path.relative_to(ROOT)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in ANALYZED_FILES
}
print('audit_timestamp=', AUDIT_TIMESTAMP)
print('tool_versions=', TOOL_VERSIONS)
print('read_only_commands_used=')
for command in READ_ONLY_COMMANDS_USED:
    print(' -', command)
print('sha256_analyzed_files=')
for path, digest in FILE_SHA256.items():
    print(path, digest)

In [ ]:
TRACEABILITY = [
 {'finding_id':'F-001','type':'CONFIRMED BY CODE','evidence_location':'RAG-evaluation.ipynb cells 15/21/26; app/retrieval.py:48/100/215','diagnostic_test':'AST call-path validation','result':'Distinct strategy functions','recommendation':'Keep dispatch; inspect effective candidates'},
 {'finding_id':'F-002','type':'CONFIRMED BY EXECUTION','evidence_location':'Three stored retrieval result CSVs','diagnostic_test':'Pairwise top-5 sequence comparison','result':'Semantic/hybrid identical 30/31','recommendation':'Inspect lexical candidate/score distributions and RRF sensitivity'},
 {'finding_id':'F-003','type':'CONFIRMED BY EXECUTION','evidence_location':'Three stored retrieval result CSVs','diagnostic_test':'Language-aware per-query diff','result':'12 rankings differ; 3 empty; aggregate hits cancel','recommendation':'Log detection and add validated fallback'},
 {'finding_id':'F-004','type':'CONFIRMED BY CODE/EXECUTION','evidence_location':'RAG-evaluation.ipynb evaluator cells','diagnostic_test':'Metric recomputation','result':'Hit equals Binary Recall in every row','recommendation':'Remove duplicate metric; add true recall'},
 {'finding_id':'F-005','type':'CONFIRMED BY EXECUTION','evidence_location':'gold_references.csv + stored rankings','diagnostic_test':'Independent evidence-aware metrics','result':'Doc Hit .774 vs true recall .611; coverage .430 for semantic k5','recommendation':'Report coverage, nDCG and MAP'},
 {'finding_id':'F-006','type':'INCONCLUSIVE','evidence_location':'Hybrid CSVs omit candidates/raw scores','diagnostic_test':'Guarded live candidate probe','result':'NOT RUN','recommendation':'Execute read-only after authorization'},
 {'finding_id':'F-007','type':'CONFIRMED methodological limitation','evidence_location':'31 questions / 77 evidences','diagnostic_test':'Bootstrap + paired sign-flip tests','result':'Wide intervals; small effects','recommendation':'Expand with predeclared discriminatory queries'},
]
EXECUTION_MANIFEST = [
 {'section':'Environment metadata','status':'EXECUTED OFFLINE DURING AUDIT','writes':False},
 {'section':'Static call-path validation','status':'EXECUTED OFFLINE DURING AUDIT','writes':False},
 {'section':'Baseline reproduction from stored CSVs','status':'EXECUTED OFFLINE DURING AUDIT — exact match','writes':False},
 {'section':'Pairwise/query-level comparisons','status':'EXECUTED OFFLINE DURING AUDIT','writes':False},
 {'section':'Independent metrics k=1,3,5','status':'EXECUTED OFFLINE DURING AUDIT','writes':False},
 {'section':'Semantic stored aggregate k=10','status':'READ FROM EXISTING RESULT','writes':False},
 {'section':'Hybrid/language-aware k=10','status':'NOT RUN / NOT STORED','writes':False},
 {'section':'Bootstrap/permutation analysis','status':'EXECUTED OFFLINE DURING AUDIT','writes':False},
 {'section':'Empty-result fallback simulation','status':'EXECUTED OFFLINE COUNTERFACTUAL','writes':False},
 {'section':'Live database candidate probe','status':'NOT RUN','writes':False},
 {'section':'Model loading / embedding','status':'NOT RUN','writes':False},
 {'section':'Paid APIs','status':'NOT RUN','writes':False},
]
print('TRACEABILITY')
for row in TRACEABILITY:print(row)
print('EXECUTION MANIFEST')
for row in EXECUTION_MANIFEST:print(row)